# 05 Language-model coding of the explanations
Registered plan, Section 7.5. Primary coder: `gpt-oss-120b` (open weights) in vLLM's batch-invariant mode with a fixed seed, on a Colab GPU with about 80 GB (otherwise `gpt-oss-20b`, decided before coding). Check coders: Claude (`claude-opus-5-5`, Message Batches API) and Gemini, three runs each, majority code. Secrets: `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`.

Before the test set is coded, the primary coder must be checked on the development set: two runs must give identical codes, and agreement with the human coders must be computed.

*Generated by `_build_notebooks.py`; edit that file, not this notebook.*

In [1]:
# --- Setup -------------------------------------------------------------
# MODE = "simulated" runs on the synthetic data (stand-ins replace the GPU models
# and APIs). MODE = "real" runs on the de-identified data. Until
# Analysis/text_frozen/text_config.json has frozen = true, real mode processes
# only the development set (plan, Section 7.1).
#
# In Colab, the code is a fresh clone of the GitHub repository, so the freeze
# check can verify the commit. The data are read from, and the results written
# to, the copy of the repository in your Google Drive. Its folder, relative to
# My Drive, is the Colab Secret DRIVE_FOLDER (kept out of this public file).
MODE = "simulated"
REPO_URL = "https://github.com/DGK-Law-and-Cognition-Lab/Moral-Authority-in-Copyright.git"
CLONE = "/content/Moral-Authority-in-Copyright"

import os, sys, json, subprocess
assert MODE in ("simulated", "real")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    try:
        DRIVE_COPY = os.path.join("/content/drive/MyDrive", userdata.get("DRIVE_FOLDER").strip().strip("/"))
    except Exception as e:
        raise RuntimeError("Add the Colab Secret DRIVE_FOLDER (the Drive copy's folder, relative to My Drive) "
                           "and turn on this notebook's access to it.") from e
    if not os.path.isdir(os.path.join(DRIVE_COPY, "Analysis", "output")):
        raise FileNotFoundError("DRIVE_FOLDER does not point to a copy of the repository (no Analysis/output there).")
    if not os.path.exists(CLONE):
        subprocess.run(["git", "clone", "--quiet", REPO_URL, CLONE], check=True)
    os.chdir(CLONE)
sys.path.insert(0, os.path.join(os.getcwd() if IN_COLAB else os.path.abspath(".."), "python"))
sys.path.insert(0, os.path.abspath(os.path.join("Analysis", "python")))
import textlib as tl
ROOT = tl.repo_root()
if IN_COLAB:
    tl.link_drive_outputs(ROOT, os.path.join(DRIVE_COPY, "Analysis", "output"), [MODE])
CFG = tl.load_config(ROOT)
SEED = CFG["seed"]
COMMIT = subprocess.run(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository folder:", os.path.basename(ROOT), "| code commit:", COMMIT, "| mode:", MODE,
      "| text config frozen:", CFG["frozen"])


Repository folder: Moral-Authority-in-Copyright | code commit: 5a92b04 | mode: simulated | text config frozen: False


In [2]:
import numpy as np, pandas as pd
df_all = tl.load_text_input(ROOT, MODE)
df = df_all[df_all["text_ok"]].copy()
if MODE == "real" and not CFG["frozen"]:
    df = df[df["text_set"] == "dev"].copy()
    print("Text configuration not frozen: DEVELOPMENT SET ONLY.")
if MODE == "real" and CFG["frozen"]:
    FREEZE = tl.freeze_check(ROOT, CFG)   # frozen files match their recorded SHA-256 and are committed
    print("Frozen configuration verified at commit", FREEZE["commit"])
print(len(df), "explanations with at least", CFG["min_words"], "words;",
      (df.text_set == "dev").sum(), "development,", (df.text_set == "test").sum(), "test.")


545 explanations with at least 3 words; 295 development, 250 test.


In [3]:
CODEBOOK = tl.codebook_prompt(ROOT)
lc = CFG["llm_coders"]
out = tl.output_dir(ROOT, MODE, "coding")
items = df[["pid", "Explain_Ruling"]].reset_index(drop=True)
prompts = [tl.build_prompt(CODEBOOK, t) for t in items.Explain_Ruling]
def get_secret(name):
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    return os.environ.get(name)
def save_runs(key, runs):   # runs[r][i] = raw output string (or None)
    with open(os.path.join(out, f"runs_{key}.jsonl"), "w") as f:
        for i, pid in enumerate(items.pid):
            f.write(json.dumps({"pid": pid, "runs": [r[i] for r in runs]}) + "\n")
def parsed(raw):
    try:
        return tl.parse_coding(raw) if raw is not None else None
    except ValueError:
        return None
print(len(prompts), "items to code")

545 items to code


## Primary coder: gpt-oss in vLLM
Skipped in simulated mode (the keyword stand-in `tl.simulated_coder` is used instead).

In [4]:
p = lc["primary"]
if MODE == "simulated":
    runs_primary = [[json.dumps(tl.simulated_coder(t)) for t in items.Explain_Ruling]]
else:
    os.environ["VLLM_BATCH_INVARIANT"] = "1" if p["batch_invariant"] else "0"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], check=True)
    from vllm import LLM, SamplingParams
    try:   # newer vLLM
        from vllm.sampling_params import StructuredOutputsParams
        so = {"structured_outputs": StructuredOutputsParams(json=tl.CODING_SCHEMA)}
    except ImportError:   # older vLLM
        from vllm.sampling_params import GuidedDecodingParams
        so = {"guided_decoding": GuidedDecodingParams(json=tl.CODING_SCHEMA)}
    MODEL_ID = p["id"]   # change to p["fallback_id"] only if the GPU lacks ~80 GB; record the choice before coding
    if CFG["frozen"]:
        assert p["revision"], "pin the gpt-oss revision in text_config.json before freezing"
    llm = LLM(model=MODEL_ID, revision=p["revision"], seed=p["seed"])
    sp = SamplingParams(temperature=p["temperature"], seed=p["seed"], max_tokens=p["max_tokens"], **so)
    msgs = [[{"role": "user", "content": q}] for q in prompts]
    res = llm.chat(msgs, sp, chat_template_kwargs={"reasoning_effort": "low"})
    runs_primary = [[r.outputs[0].text for r in res]]
    # Determinism check (Section 7.5): a second pass must give identical output.
    res2 = llm.chat(msgs, sp, chat_template_kwargs={"reasoning_effort": "low"})
    same = [r1.outputs[0].text == r2.outputs[0].text for r1, r2 in zip(res, res2)]
    json.dump({"model": MODEL_ID, "items": len(same), "identical": int(sum(same))},
              open(os.path.join(out, "determinism_gptoss.json"), "w"), indent=2)
    print("identical on rerun:", sum(same), "of", len(same))
    assert all(same), "the primary coder is not deterministic; fix the vLLM settings before coding the test set"
save_runs(p["key"], runs_primary)
codes_primary = [parsed(x) for x in runs_primary[0]]
print("valid:", sum(c is not None for c in codes_primary), "of", len(codes_primary))

valid: 545 of 545


## Check coder: Claude (Message Batches API)
Structured output with the coding schema; three runs; refusals and errors are recorded as failed runs.

In [5]:
c = [x for x in lc["checks"] if x["key"] == "claude"][0]
if MODE == "simulated":
    runs_claude = [[json.dumps(tl.simulated_coder(t)) for t in items.Explain_Ruling] for _ in range(c["runs"])]
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
    import time, anthropic
    from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
    from anthropic.types.messages.batch_create_params import Request
    client = anthropic.Anthropic(api_key=get_secret("ANTHROPIC_API_KEY"))
    runs_claude = []
    for r in range(c["runs"]):
        batch = client.messages.batches.create(requests=[
            Request(custom_id=f"i{i}_r{r}", params=MessageCreateParamsNonStreaming(
                model=c["id"], max_tokens=c["max_tokens"],
                output_config={"effort": c["effort"], "format": {"type": "json_schema", "schema": tl.CODING_SCHEMA}},
                messages=[{"role": "user", "content": q}]))
            for i, q in enumerate(prompts)])
        while client.messages.batches.retrieve(batch.id).processing_status != "ended":
            time.sleep(60)
        got = {}
        for res in client.messages.batches.results(batch.id):
            i = int(res.custom_id.split("_")[0][1:])
            if res.result.type == "succeeded" and res.result.message.stop_reason != "refusal":
                got[i] = next((b.text for b in res.result.message.content if b.type == "text"), None)
        runs_claude.append([got.get(i) for i in range(len(prompts))])
save_runs(c["key"], runs_claude)

## Check coder: Gemini
Structured JSON output; three runs. Set the model ID in `text_config.json` before coding.

In [6]:
g = [x for x in lc["checks"] if x["key"] == "gemini"][0]
if MODE == "simulated":
    runs_gemini = [[json.dumps(tl.simulated_coder(t)) for t in items.Explain_Ruling] for _ in range(g["runs"])]
else:
    assert g["id"], "set the Gemini model ID in text_config.json before coding"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai"], check=True)
    from google import genai
    from google.genai import types
    gc = genai.Client(api_key=get_secret("GEMINI_API_KEY"))
    def gem(q):
        try:
            cfg = types.GenerateContentConfig(response_mime_type="application/json", response_json_schema=tl.CODING_SCHEMA)
        except TypeError:   # older google-genai
            cfg = types.GenerateContentConfig(response_mime_type="application/json", response_schema=tl.CODING_SCHEMA)
        try:
            return gc.models.generate_content(model=g["id"], contents=q, config=cfg).text
        except Exception as e:
            print("failed:", e); return None
    runs_gemini = [[gem(q) for q in prompts] for _ in range(g["runs"])]
save_runs(g["key"], runs_gemini)

## Majority codes, agreement across runs, and agreement between coders

In [7]:
all_runs = {"gptoss": runs_primary, "claude": runs_claude, "gemini": runs_gemini}
maj = {}
for key, runs in all_runs.items():
    per_item = [[parsed(r[i]) for r in runs] for i in range(len(items))]
    if len(runs) > 1:
        tl.run_agreement(per_item).to_csv(os.path.join(out, f"run_agreement_{key}.csv"), index=False)
    m = [tl.majority(x) for x in per_item]
    frame = pd.DataFrame([x if x else {} for x in m]); frame.insert(0, "pid", items.pid.values)
    frame.to_csv(os.path.join(out, f"codes_{key}.csv"), index=False)
    maj[key] = frame
fields = tl.REASON_CODES + ["stance"]
rows = []
for a, b in (("gptoss", "claude"), ("gptoss", "gemini"), ("claude", "gemini")):
    x, y = maj[a].set_index("pid"), maj[b].set_index("pid")
    common = x.dropna(subset=fields).index.intersection(y.dropna(subset=fields).index)
    for f in fields:
        k = tl.kappa(x.loc[common, f], y.loc[common, f], categorical=(f == "stance"))
        rows.append({"coder_a": a, "coder_b": b, "field": f, "n": len(common), "kappa": k})
pd.DataFrame(rows).to_csv(os.path.join(out, "coder_agreement.csv"), index=False)
pd.DataFrame(rows).pivot_table(index="field", columns=["coder_a", "coder_b"], values="kappa").round(2)

coder_a           claude gptoss       
coder_b           gemini claude gemini
field                                 
credit               1.0    1.0    1.0
fairness_moral       1.0    1.0    1.0
fame_status_power    1.0    1.0    1.0
law_rules            1.0    1.0    1.0
market_effect        1.0    1.0    1.0
nonmonetary_harm     1.0    1.0    1.0
permission           1.0    1.0    1.0
profit               1.0    1.0    1.0
remedy               1.0    1.0    1.0
stance               1.0    1.0    1.0
transformation       1.0    1.0    1.0

## Reliability gate (Section 7.5)
Development set: kappa between the two humans (independent codes, `human_codes_dev.csv`) and between each model and the resolved human consensus (`human_codes_dev_consensus.csv`). Test set: kappa between one human (`human_codes_test.csv`, 100 random test responses) and the primary coder. A reason code is retained only if the humans agree (kappa >= .70), the primary coder agrees with the consensus (>= .70), and, for test-set analyses, the primary coder agrees with the test-set human (>= .70). Stance is reported but is not a reason code.

In [8]:
tf = lambda name: os.path.join(ROOT, "Analysis", "text_frozen", name)
dev_ind, dev_con, test_h = tf("human_codes_dev.csv"), tf("human_codes_dev_consensus.csv"), tf("human_codes_test.csv")
ret = {"retained": [], "dev_gate_applied": False, "test_gate_applied": False}
if os.path.exists(dev_ind) and os.path.exists(dev_con):
    rd = lambda f: pd.read_csv(f, dtype={"pid": str})
    dev_rel = tl.reliability_dev(rd(dev_ind), rd(dev_con), maj)
    dev_rel.to_csv(os.path.join(out, "reliability_dev.csv"), index=False); display(dev_rel)
    test_rel = None
    if os.path.exists(test_h) and (MODE == "simulated" or CFG["frozen"]):
        test_rel = tl.reliability_test(rd(test_h), maj["gptoss"])
        test_rel.to_csv(os.path.join(out, "reliability_test.csv"), index=False); display(test_rel)
    ret = {"retained": tl.retained_categories(dev_rel, test_rel), "dev_gate_applied": True,
           "test_gate_applied": test_rel is not None}
elif MODE == "simulated":
    ret = {"retained": tl.REASON_CODES, "dev_gate_applied": False, "test_gate_applied": False,
           "note": "SIMULATED: no human codes; all reason codes marked retained for testing"}
else:
    print("No human codes yet: no category can be retained (plan, Section 7.5).")
json.dump(ret, open(os.path.join(out, "retained_categories.json"), "w"), indent=2)
ret

{'retained': ['permission',
  'transformation',
  'nature_of_work',
  'amount_used',
  'market_effect',
  'profit',
  'credit',
  'nonmonetary_harm',
  'fame_status_power',
  'fairness_moral',
  'law_rules',
  'remedy'],
 'dev_gate_applied': False,
 'test_gate_applied': False,
 'note': 'SIMULATED: no human codes; all reason codes marked retained for testing'}